# CS 129 — Predicting Stock Volatility from SEC 10-K Filing Language
## Experiments Notebook

**Run order:**
1. Data pipeline (once — ~30-60 min for dev subset)
2. Feature engineering
3. Model training with custom hyperparameter loops
4. Evaluation & figures → fills in Section 4 of the milestone doc

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print('PyTorch:', torch.__version__)
device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print('Device:', device)

PyTorch: 2.9.0
Device: mps


---
## Step 1 — Data Collection
Fetches 10-K Item 1A text, realized volatility, and financial ratios.
**Only needs to run once.** Results are cached to `Data/raw/`.

In [2]:
from src.data_pipeline import run_pipeline

raw_dir = '../Data/raw'
if not os.path.exists(os.path.join(raw_dir, 'filings.csv')):
    run_pipeline()
else:
    print('Raw data already collected — skipping pipeline.')

Raw data already collected — skipping pipeline.


In [3]:
# Quick sanity check
df_filings    = pd.read_csv('../Data/raw/filings.csv')
df_targets    = pd.read_csv('../Data/raw/targets.csv')
df_financials = pd.read_csv('../Data/raw/financials.csv')

print(f'Filings    : {len(df_filings)} rows')
print(f'Targets    : {len(df_targets)} rows')
print(f'Financials : {len(df_financials)} rows')
print()
print(df_targets[['volatility','high_volatility']].describe())

Filings    : 304 rows
Targets    : 304 rows
Financials : 304 rows

       volatility  high_volatility
count  304.000000       304.000000
mean     0.282551         0.342105
std      0.147751         0.475197
min      0.122527         0.000000
25%      0.186602         0.000000
50%      0.239652         0.000000
75%      0.329728         1.000000
max      1.344480         1.000000


---
## Step 2 — Feature Engineering

In [4]:
from src.features import build_all_features

feature_sets, targets, df_train, df_val = build_all_features()

y_reg_train, y_reg_val   = targets['regression']
y_cls_train, y_cls_val   = targets['classification']

print(f'\nTrain size : {len(df_train)}')
print(f'Val size   : {len(df_val)}')
print(f'High-vol rate (train): {y_cls_train.mean():.1%}')
print(f'High-vol rate (val)  : {y_cls_val.mean():.1%}')

Loading raw data...
  Train: 216 | Val: 88

Building X_financial...
Building X_sentiment...


  Sentiment:   0%|          | 0/216 [00:00<?, ?it/s]

  Sentiment:   9%|▉         | 20/216 [00:00<00:01, 192.42it/s]

  Sentiment:  19%|█▊        | 40/216 [00:00<00:01, 120.37it/s]

  Sentiment:  25%|██▌       | 54/216 [00:00<00:01, 100.76it/s]

...

  Sentiment:  70%|███████   | 62/88 [00:00<00:00, 124.73it/s]

  Sentiment:  89%|████████▊ | 78/88 [00:00<00:00, 134.09it/s]

Building X_tfidf...


Building X_finbert...
  Loading FinBERT embeddings from cache...



Feature shapes (train):
  financial   : (216, 4)
  sentiment   : (216, 8)
  tfidf       : (216, 500)
  finbert     : (216, 768)
  all         : (216, 1280)

Train size : 216
Val size   : 88
High-vol rate (train): 25.0%
High-vol rate (val)  : 56.8%


In [5]:
from src.evaluate import plot_target_distribution

threshold = df_targets['vol_threshold'].iloc[0]
plot_target_distribution(y_reg_train, y_reg_val, threshold)

  Saved → /Users/juansandoval/Desktop/CS129Final/results/figures/target_distribution.png


---
## Step 3 — Model Training

### 3a. Ridge Regression (volatility prediction)

In [6]:
from src.models import ridge_grid_search
from src.evaluate import plot_regularization_curves

ridge_results  = []
ridge_models   = {}

for fs_name, (X_tr, X_val) in feature_sets.items():
    print(f'Ridge  [{fs_name}]')
    df_res, best_model = ridge_grid_search(
        X_tr, y_reg_train, X_val, y_reg_val, feature_set_name=fs_name
    )
    ridge_results.append(df_res)
    ridge_models[fs_name] = best_model

ridge_df = pd.concat(ridge_results, ignore_index=True)

# Best per feature set
print('\nBest val RMSE per feature set:')
print(ridge_df.loc[ridge_df.groupby('feature_set')['val_rmse'].idxmin(),
                   ['feature_set','alpha','val_rmse','val_r2']].to_string(index=False))

Ridge  [financial]
Ridge  [sentiment]
Ridge  [tfidf]
Ridge  [finbert]
Ridge  [all]

Best val RMSE per feature set:
feature_set   alpha  val_rmse    val_r2
        all  100.00  0.247983 -0.409600
  financial    0.01  0.252205 -0.458004
    finbert  100.00  0.247973 -0.409492
  sentiment 1000.00  0.251954 -0.455110
      tfidf    0.10  0.242450 -0.347405


In [7]:
plot_regularization_curves(
    ridge_df, param_col='alpha', metric_col='val_rmse',
    title='Ridge Regression — RMSE vs Alpha', fname='ridge_rmse_curves.png'
)

  Saved → /Users/juansandoval/Desktop/CS129Final/results/figures/ridge_rmse_curves.png


### 3b. Logistic Regression (high/low volatility classification)

In [8]:
from src.models import logreg_grid_search

logreg_results = []
logreg_models  = {}

for fs_name, (X_tr, X_val) in feature_sets.items():
    print(f'LogReg [{fs_name}]')
    df_res, best_model = logreg_grid_search(
        X_tr, y_cls_train, X_val, y_cls_val, feature_set_name=fs_name
    )
    logreg_results.append(df_res)
    logreg_models[fs_name] = best_model

logreg_df = pd.concat(logreg_results, ignore_index=True)

print('\nBest val AUC per feature set:')
print(logreg_df.loc[logreg_df.groupby('feature_set')['val_auc'].idxmax(),
                    ['feature_set','C','val_auc','val_f1']].to_string(index=False))

LogReg [financial]
LogReg [sentiment]
LogReg [tfidf]
LogReg [finbert]
LogReg [all]



Best val AUC per feature set:
feature_set       C  val_auc   val_f1
        all   0.010 0.647895 0.500000
  financial   0.001 0.500000 0.724638
    finbert   0.001 0.648421 0.487179
  sentiment   1.000 0.531579 0.505747
      tfidf 100.000 0.696842 0.582278


In [9]:
plot_regularization_curves(
    logreg_df, param_col='C', metric_col='val_auc',
    title='Logistic Regression — AUC vs C', fname='logreg_auc_curves.png'
)

  Saved → /Users/juansandoval/Desktop/CS129Final/results/figures/logreg_auc_curves.png


### 3c. Random Forest

In [10]:
from src.models import rf_grid_search
from src.evaluate import plot_feature_importance

rf_results = []
rf_models  = {}

for fs_name, (X_tr, X_val) in feature_sets.items():
    print(f'RF     [{fs_name}]')
    df_res, best_model = rf_grid_search(
        X_tr, y_cls_train, X_val, y_cls_val,
        task='classification', feature_set_name=fs_name
    )
    rf_results.append(df_res)
    rf_models[fs_name] = best_model

rf_df = pd.concat(rf_results, ignore_index=True)

print('\nBest val AUC per feature set:')
print(rf_df.loc[rf_df.groupby('feature_set')['val_auc'].idxmax(),
                ['feature_set','n_estimators','max_depth','min_samples_leaf',
                 'val_auc','val_f1']].to_string(index=False))

RF     [financial]


RF (financial, classification):   0%|          | 0/18 [00:00<?, ?it/s]

RF (financial, classification):  11%|█         | 2/18 [00:00<00:01, 10.55it/s]

RF (financial, classification):  22%|██▏       | 4/18 [00:00<00:01, 10.55it/s]

RF (financial, classification):  33%|███▎      | 6/18 [00:00<00:01, 10.48it/s]

...

RF (all, classification):  83%|████████▎ | 15/18 [00:01<00:00,  6.90it/s]

RF (all, classification):  89%|████████▉ | 16/18 [00:02<00:00,  6.61it/s]

RF (all, classification):  94%|█████████▍| 17/18 [00:02<00:00,  6.42it/s]

RF (all, classification): 100%|██████████| 18/18 [00:02<00:00,  6.39it/s]


Best val AUC per feature set:
feature_set  n_estimators max_depth  min_samples_leaf  val_auc   val_f1
        all           100        20                 1 0.678684 0.349206
  financial           100        10                 1 0.500000 0.000000
    finbert           200        20                 1 0.667895 0.266667
  sentiment           200        20                 1 0.636316 0.241379
      tfidf           200        10                 1 0.679737 0.295082


In [11]:
# Feature importance for the best RF on the 'all' feature set
best_rf = rf_models.get('all') or rf_models.get('finbert')

# Build feature names for the combined feature set
fin_names  = ['debt_equity','roa','current_ratio','log_mktcap']
sent_names = ['neg%','pos%','uncertainty%','litigious%',
              'strong_modal%','weak_modal%','fog_index','log_word_count']
tfidf_names = [f'tfidf_{i}' for i in range(500)]
fb_names    = [f'finbert_{i}' for i in range(768)]
all_names   = fin_names + sent_names + tfidf_names + fb_names

if best_rf is not None and hasattr(best_rf, 'feature_importances_'):
    n_feat = len(best_rf.feature_importances_)
    names  = all_names[:n_feat] if n_feat == len(all_names) else [f'feat_{i}' for i in range(n_feat)]
    plot_feature_importance(best_rf, names, top_n=20)

  Saved → /Users/juansandoval/Desktop/CS129Final/results/figures/rf_importance.png


### 3d. Custom PyTorch MLP (manual training loop)

Full forward / backward pass implemented in `src/models.py::train_mlp`.  
Grid search sweeps 24 hyperparameter combinations per feature set.

In [12]:
from src.models import mlp_grid_search
from src.evaluate import plot_training_curves

mlp_results = []
mlp_models  = {}
mlp_histories = {}

# Run MLP on key feature sets (skip 'all' if memory is tight)
mlp_target_sets = ['financial', 'sentiment', 'tfidf', 'finbert']

for fs_name in mlp_target_sets:
    X_tr, X_val = feature_sets[fs_name]
    print(f'\nMLP    [{fs_name}]  input_dim={X_tr.shape[1]}')
    df_res, best_model, best_hist = mlp_grid_search(
        X_tr, y_cls_train, X_val, y_cls_val,
        task='classification', feature_set_name=fs_name
    )
    mlp_results.append(df_res)
    mlp_models[fs_name]    = best_model
    mlp_histories[fs_name] = best_hist

mlp_df = pd.concat(mlp_results, ignore_index=True)

print('\nBest val AUC per feature set (MLP):')
print(mlp_df.loc[mlp_df.groupby('feature_set')['val_auc'].idxmax(),
                 ['feature_set','hidden_dims','dropout','lr',
                  'epochs_trained','val_auc','val_f1']].to_string(index=False))


MLP    [financial]  input_dim=4


MLP grid (financial, classification):   0%|          | 0/36 [00:00<?, ?it/s]

MLP grid (financial, classification):   3%|▎         | 1/36 [00:00<00:26,  1.31it/s]

MLP grid (financial, classification):   6%|▌         | 2/36 [00:00<00:15,  2.26it/s]

MLP grid (financial, classification):   8%|▊         | 3/36 [00:01<00:10,  3.07it/s]

...

MLP grid (finbert, classification):  94%|█████████▍| 34/36 [00:10<00:00,  2.73it/s]

MLP grid (finbert, classification):  97%|█████████▋| 35/36 [00:11<00:00,  3.00it/s]

MLP grid (finbert, classification): 100%|██████████| 36/36 [00:11<00:00,  3.15it/s]

MLP grid (finbert, classification): 100%|██████████| 36/36 [00:11<00:00,  3.16it/s]


Best val AUC per feature set (MLP):
feature_set     hidden_dims  dropout     lr  epochs_trained  val_auc   val_f1
  financial  [256, 128, 64]      0.2 0.0001              13 0.527368 0.145455
    finbert  [256, 128, 64]      0.3 0.0010              12 0.693684 0.680851
  sentiment  [256, 128, 64]      0.2 0.0010              13 0.603684 0.586957
      tfidf [512, 256, 128]      0.3 0.0003              25 0.722632 0.343750


In [13]:
# Training curves for best MLP feature set
best_fs   = mlp_df.loc[mlp_df['val_auc'].idxmax(), 'feature_set']
best_hist = mlp_histories[best_fs]
plot_training_curves(
    best_hist,
    title=f'MLP Training Curves — {best_fs} features',
    fname='mlp_training_curves.png'
)
print(f'Best MLP feature set: {best_fs}')
print(f'Epochs trained      : {best_hist["epochs_trained"]}')

  Saved → /Users/juansandoval/Desktop/CS129Final/results/figures/mlp_training_curves.png
Best MLP feature set: tfidf
Epochs trained      : 25


---
## Step 4 — Evaluation & Figures

In [14]:
from src.evaluate import plot_roc_curves, plot_model_comparison_heatmap, print_results_table

# ── Collect best val probabilities for ROC curves ──
model_probs = {}

# Best LogReg per feature set
best_logreg_fs = logreg_df.loc[logreg_df['val_auc'].idxmax(), 'feature_set']
best_logreg    = logreg_models[best_logreg_fs]
X_val_lr       = feature_sets[best_logreg_fs][1]
model_probs[f'LogReg ({best_logreg_fs})'] = best_logreg.predict_proba(X_val_lr)[:, 1]

# Best RF
best_rf_fs = rf_df.loc[rf_df['val_auc'].idxmax(), 'feature_set']
best_rf_m  = rf_models[best_rf_fs]
X_val_rf   = feature_sets[best_rf_fs][1]
model_probs[f'RF ({best_rf_fs})'] = best_rf_m.predict_proba(X_val_rf)[:, 1]

# Best MLP — move to CPU for inference so tensor device always matches
best_mlp_row = mlp_df.loc[mlp_df['val_auc'].idxmax()]
best_mlp_fs  = best_mlp_row['feature_set']
best_mlp_m   = mlp_models[best_mlp_fs].cpu()
X_val_mlp    = torch.tensor(feature_sets[best_mlp_fs][1], dtype=torch.float32)
best_mlp_m.eval()
with torch.no_grad():
    logits = best_mlp_m(X_val_mlp).numpy()
model_probs[f'MLP ({best_mlp_fs})'] = 1 / (1 + np.exp(-logits))

plot_roc_curves(model_probs, y_cls_val)

  Saved → /Users/juansandoval/Desktop/CS129Final/results/figures/roc_curves.png


In [15]:
# ── Summary table (Section 4 content) ──
summary_rows = []

for fs_name in feature_sets:
    # LogReg
    sub = logreg_df[logreg_df['feature_set'] == fs_name]
    if len(sub):
        best = sub.loc[sub['val_auc'].idxmax()]
        summary_rows.append({'model': 'LogReg', 'feature_set': fs_name,
                              'val_auc': best['val_auc'], 'val_f1': best['val_f1']})
    # RF
    sub = rf_df[rf_df['feature_set'] == fs_name]
    if len(sub):
        best = sub.loc[sub['val_auc'].idxmax()]
        summary_rows.append({'model': 'RF', 'feature_set': fs_name,
                              'val_auc': best['val_auc'], 'val_f1': best['val_f1']})
    # MLP
    sub = mlp_df[mlp_df['feature_set'] == fs_name]
    if len(sub):
        best = sub.loc[sub['val_auc'].idxmax()]
        summary_rows.append({'model': 'MLP', 'feature_set': fs_name,
                              'val_auc': best['val_auc'], 'val_f1': best['val_f1']})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('../results/val_summary.csv', index=False)

 model feature_set  val_auc   val_f1
LogReg   financial 0.500000 0.724638
    RF   financial 0.500000 0.000000
   MLP   financial 0.527368 0.145455
LogReg   sentiment 0.531579 0.505747
    RF   sentiment 0.636316 0.241379
   MLP   sentiment 0.603684 0.586957
LogReg       tfidf 0.696842 0.582278
    RF       tfidf 0.679737 0.295082
   MLP       tfidf 0.722632 0.343750
LogReg     finbert 0.648421 0.487179
    RF     finbert 0.667895 0.266667
   MLP     finbert 0.693684 0.680851
LogReg         all 0.647895 0.500000
    RF         all 0.678684 0.349206


In [16]:
from src.evaluate import plot_model_comparison_heatmap
plot_model_comparison_heatmap(summary_df, metric='val_auc')

  Saved → /Users/juansandoval/Desktop/CS129Final/results/figures/model_comparison.png


---
## Section 4 Key Findings (fill in after running)

```
• Best classification model:  ________  (val AUC = ____)
• Best feature set:           ________
• Text features vs financial-only AUC gain:  +____
• FinBERT vs TF-IDF AUC:     ____ vs ____
• Best regression model:      ________  (val RMSE = ____)
• MLP early stopped at epoch: ____  (best = ____)
```